# 01. Explore WANDS

Goal: know the data before indexing it. Counts, label split, category tree, and the format of `product_features`.

In [1]:
import pandas as pd

DATA = '../data/WANDS/dataset'
products = pd.read_csv(f'{DATA}/product.csv', sep='\t')
queries = pd.read_csv(f'{DATA}/query.csv', sep='\t')
labels = pd.read_csv(f'{DATA}/label.csv', sep='\t')

for name, df in [('products', products), ('queries', queries), ('labels', labels)]:
    print(name, df.shape)

products (42994, 9)
queries (480, 3)
labels (233448, 4)


## Products: columns and missing values

In [2]:
products.info()
products.isna().mean().round(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42994 entries, 0 to 42993
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           42994 non-null  int64  
 1   product_name         42994 non-null  object 
 2   product_class        40142 non-null  object 
 3   category hierarchy   41438 non-null  object 
 4   product_description  36986 non-null  object 
 5   product_features     42994 non-null  object 
 6   rating_count         33542 non-null  float64
 7   average_rating       33542 non-null  float64
 8   review_count         33542 non-null  float64
dtypes: float64(3), int64(1), object(5)
memory usage: 3.0+ MB


product_id             0.000
product_name           0.000
product_class          0.066
category hierarchy     0.036
product_description    0.140
product_features       0.000
rating_count           0.220
average_rating         0.220
review_count           0.220
dtype: float64

## Labels: overall split, and per query

In [3]:
print(labels['label'].value_counts(normalize=True).round(3))

per_query = labels.pivot_table(index='query_id', columns='label', values='id', aggfunc='count', fill_value=0)
per_query['total'] = per_query.sum(axis=1)
per_query.describe().round(1)

Partial       0.628
Irrelevant    0.262
Exact         0.110
Name: label, dtype: float64


label,Exact,Irrelevant,Partial,total
count,480.0,480.0,480.0,480.0
mean,53.4,127.5,305.5,486.4
std,105.5,346.6,504.0,708.5
min,0.0,0.0,0.0,1.0
25%,1.0,3.0,67.8,122.8
50%,10.0,18.5,126.0,216.5
75%,63.0,73.2,294.5,523.8
max,878.0,2940.0,3438.0,4329.0


In [4]:
# Queries with zero Exact labels. Strict precision@10 can never be above 0 for these.
no_exact = per_query[per_query.get('Exact', 0) == 0]
print(len(no_exact), 'queries have no Exact label')
queries[queries.query_id.isin(no_exact.index)].head(10)

101 queries have no Exact label


,query_id,query,query_class
19,19,gurney slade 56,Chandeliers
23,23,stoneford end tables white and wood,End Tables
47,47,porcelain loaf pan,Bread & Loaf Pans
49,49,moen matte black hooks,Towel & Robe Hooks
52,52,westling coffee table,Coffee & Cocktail Tables
53,53,carolyn console table,Sofa & Console Tables
56,56,royal blue counter height chairs,Bar Stools
67,67,osgood mirror,Wall & Accent Mirrors
73,73,certified international melamine,Plates & Saucers
79,79,palram harmony greenhouses,Awnings


## Category hierarchy

In [5]:
levels = products['category hierarchy'].dropna().str.split(' / ')
print('depth:'); print(levels.str.len().value_counts().sort_index())
print('distinct level-1:', levels.str[0].nunique())
print(levels.str[0].value_counts().head(10))

depth:
1      103
2     1720
3    16297
4    16256
5     6453
6      609
Name: category hierarchy, dtype: int64
distinct level-1: 55
Furniture                 16039
Home Improvement           4686
Décor & Pillows            4612
Outdoor                    3394
Storage & Organization     2175
Lighting                   2072
Rugs                       2002
Bed & Bath                 1865
Kitchen & Tabletop         1615
Baby & Kids                1204
Name: category hierarchy, dtype: int64


## Product features format

Pipe-separated `key : value` pairs. On Day 3 each feature becomes one "keyword" for pooling.

In [6]:
feats = products['product_features'].str.split('|')
print('features per product:'); print(feats.str.len().describe().round(1))
print()
for f in feats.iloc[0][:8]:
    print(repr(f))

features per product:
count    42994.0
mean        56.6
std         24.9
min          1.0
25%         38.0
50%         55.0
75%         73.0
max        403.0
Name: product_features, dtype: float64

'overallwidth-sidetoside:64.7'
'dsprimaryproductstyle : modern'
'countryoforigin-additionaldetails : made in viet nam'
'headboardtype : slat'
'headboardheight-toptobedframe:25.35'
'weightcapacity:500'
'estimatedtimetosetup:120'
'overalllength-headtotoe:77.9'


## Queries

In [7]:
print(queries['query_class'].nunique(), 'query classes')
print(queries['query'].str.split().str.len().describe().round(1))
queries.sample(10, random_state=0)

188 query classes
count    480.0
mean       3.4
std        1.5
min        1.0
25%        2.0
50%        3.0
75%        4.0
max       10.0
Name: query, dtype: float64


,query_id,query,query_class
15,15,black 5 drawer dresser by guilford,Dressers & Chests
124,124,unique coffee tables,Coffee & Cocktail Tables
141,141,podium with locking cabinet,Lecterns & Podiums
263,267,dolomite subway tile,Floor & Wall Tile
170,171,nectar queen mattress,Foam and Latex Mattresses
142,142,kitchen wooden stand,Specialty Serving
399,405,queen ann style living room chair,Accent Chairs
429,435,shoe closet,Shoe Storage
113,113,led light strip,Under Cabinet Lighting
21,21,living curtains pearl,Curtains & Drapes
